In [2]:
import pandas as pd
import json
import requests
from io import StringIO

import os
from dotenv import load_dotenv

In [12]:
S3 = os.getenv("S3")
HOMELOCAL = os.getenv("HOMELOCAL")

In [14]:
full_path = f"{HOMELOCAL}\\vscode-2026\\20260323\\rearc-quest-20260323\\bls_pr_data\\pr.data.0.Current"
df_bls = pd.read_csv(full_path, sep='\t')

print("BLS Data (Top 5 rows):")
print(df_bls.head(5))


BLS Data (Top 5 rows):
   series_id          year period         value footnote_codes
0  PRS30006011        1995    Q01           2.6            NaN
1  PRS30006011        1995    Q02           2.1            NaN
2  PRS30006011        1995    Q03           0.9            NaN
3  PRS30006011        1995    Q04           0.1            NaN
4  PRS30006011        1995    Q05           1.4            NaN


In [15]:
with open(f"{HOMELOCAL}\\vscode-2026\\20260323\\rearc-quest-20260323\\quest-part-2\\population_data.json", 'r') as f:
    population = json.load(f)
df_population = pd.json_normalize(population["data"])
print("Population Data (Top 5 rows):")
print(df_population.head(15))

Population Data (Top 5 rows):
   Nation ID         Nation  Year   Population
0    01000US  United States  2013  316128839.0
1    01000US  United States  2014  318857056.0
2    01000US  United States  2015  321418821.0
3    01000US  United States  2016  323127515.0
4    01000US  United States  2017  325719178.0
5    01000US  United States  2018  327167439.0
6    01000US  United States  2019  328239523.0
7    01000US  United States  2021  331893745.0
8    01000US  United States  2022  333287562.0
9    01000US  United States  2023  334914896.0
10   01000US  United States  2024  340110990.0


In [16]:
# -----------------------------
# quest-3-2
# -----------------------------

# Filter for years 2013–2018
df_filtered = df_population[(df_population["Year"] >= 2013) & (df_population["Year"] <= 2018)]

# Calculate mean and standard deviation
mean_population = df_filtered["Population"].mean()
std_population = df_filtered["Population"].std()

print("Mean Population:", mean_population)
print("Standard Deviation:", std_population)

Mean Population: 322069808.0
Standard Deviation: 4158441.040908095


In [17]:
# -----------------------------
# quest-3-3
# -----------------------------

# Load series datasets
full_path = f"{HOMELOCAL}\\vscode-2026\\20260323\\rearc-quest-20260323\\bls_pr_data\\pr.series"
df_series = pd.read_csv(full_path, sep='\t')
df_series.columns = df_series.columns.str.strip()

# Keep only quarterly periods (Q01–Q04)
df_bls.columns = df_bls.columns.str.strip()
df_bls = df_bls[df_bls["period"].str.startswith("Q")]
print(df_bls.head)

# Convert value to numeric
df_bls["value"] = pd.to_numeric(df_bls["value"], errors="coerce")
# Aggregate: sum per year per series_id
df_yearly = (
    df_bls
    .groupby(["series_id", "year"], as_index=False)["value"]
    .sum()
)
# Find best year per series_id
idx = df_yearly.groupby("series_id")["value"].idxmax()

df_best = df_yearly.loc[idx].reset_index(drop=True)

# Rename columns for clarity
df_best = df_best.rename(columns={
    "year": "best_year",
    "value": "total_value"
})
df_best = df_best[df_best["series_id"].isin(df_series["series_id"])]

# Final report
print(df_best.head(25))


<bound method NDFrame.head of                series_id  year period    value footnote_codes
0      PRS30006011        1995    Q01    2.600            NaN
1      PRS30006011        1995    Q02    2.100            NaN
2      PRS30006011        1995    Q03    0.900            NaN
3      PRS30006011        1995    Q04    0.100            NaN
4      PRS30006011        1995    Q05    1.400            NaN
...                  ...   ...    ...      ...            ...
37876  PRS88003203        2024    Q04  118.515            NaN
37877  PRS88003203        2024    Q05  118.125            NaN
37878  PRS88003203        2025    Q01  120.226            NaN
37879  PRS88003203        2025    Q02  120.355            NaN
37880  PRS88003203        2025    Q03  121.055            NaN

[37881 rows x 5 columns]>
            series_id  best_year  total_value
0   PRS30006011             2022       20.500
1   PRS30006012             2022       17.100
2   PRS30006013             1998      705.895
3   PRS30006021

In [18]:
# -----------------------------
# quest-3-4
# -----------------------------

# to have a CLEAN data set - reload BLS data from the file downloaded from S3
full_path = f"{HOMELOCAL}\\vscode-2026\\20260323\\rearc-quest-20260323\\bls_pr_data\\pr.data.0.Current"
df_bls = pd.read_csv(full_path, sep='\t')
df_bls.columns = df_bls.columns.str.strip()

print("BLS Data (Top 5 rows):")
print(df_bls.head(5))

with open(f"{HOMELOCAL}\\vscode-2026\\20260323\\rearc-quest-20260323\\quest-part-2\\population_data.json", 'r') as f:
    population = json.load(f)
df_population = pd.json_normalize(population["data"])
df_population.columns = df_population.columns.str.strip()

print("Population Data (Top 5 rows):")
print(df_population.head(15))

# Function to generate report
def report_value_population(series_id: str, period: str, year: int) -> pd.DataFrame:
    """
    Returns a DataFrame with value for the given series_id, period, year,
    and population for that year if available.
    """
    # Filter BLS data
    df_filtered = df_bls[
        (df_bls["series_id"].str.contains(series_id)) &
        (df_bls["period"] == period) &
        (df_bls["year"] == year)
    ].copy()
    
    if df_filtered.empty:
        print(f"No BLS data found for series_id={series_id}, period={period}, year={year}.")
        return None
    
    # Merge with population
    df_report = df_filtered.merge(
        df_population,
        left_on="year",
        right_on="Year",
        how="left"
    )
    
    # Select relevant columns
    df_report = df_report[["series_id", "year", "period", "value", "Population"]]
    
    return df_report


BLS Data (Top 5 rows):
           series_id  year period  value footnote_codes
0  PRS30006011        1995    Q01    2.6            NaN
1  PRS30006011        1995    Q02    2.1            NaN
2  PRS30006011        1995    Q03    0.9            NaN
3  PRS30006011        1995    Q04    0.1            NaN
4  PRS30006011        1995    Q05    1.4            NaN
Population Data (Top 5 rows):
   Nation ID         Nation  Year   Population
0    01000US  United States  2013  316128839.0
1    01000US  United States  2014  318857056.0
2    01000US  United States  2015  321418821.0
3    01000US  United States  2016  323127515.0
4    01000US  United States  2017  325719178.0
5    01000US  United States  2018  327167439.0
6    01000US  United States  2019  328239523.0
7    01000US  United States  2021  331893745.0
8    01000US  United States  2022  333287562.0
9    01000US  United States  2023  334914896.0
10   01000US  United States  2024  340110990.0


In [19]:
# -----------------------------
# Example usage
# -----------------------------
report = report_value_population("PRS30006032", "Q01", 2018)
print(report)


           series_id  year period  value   Population
0  PRS30006032        2018    Q01    0.5  327167439.0
